In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)


In [ ]:
from pathlib import Path
import csv
import json

import pandas as pd

from src.drive_service.logging_utils import setup_logging
from src.pipeline_paths import build_pipelines_paths
from src.turni_employee_summary import build_employee_turni_summary


In [ ]:
root = "1FUosjKncLt18JzojmX8tKQm1nbgPI133"
paths = build_pipelines_paths(root)

enriched_name = "*.enriched.csv"
enriched_files = sorted(Path(paths.enrichment_output).glob(enriched_name))
if not enriched_files:
    raise FileNotFoundError(
        f"No enriched files found in {paths.enrichment_output} with pattern {enriched_name}"
    )

paths.enrichment_output, paths.aggregation_output, len(enriched_files), enriched_files[:5]


In [ ]:
verbose = True
year_start = 2016
year_end = 2025
min_hours = None

summary_csv_path = paths.aggregation_output / "turni_employee_summary.csv"
summary_json_path = paths.aggregation_output / "turni_employee_summary.json"

setup_logging(verbose)

rows, stats = build_employee_turni_summary(
    enriched_dir=str(paths.enrichment_output),
    min_hours=min_hours,
    year_start=year_start,
    year_end=year_end,
)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(summary_csv_path, index=False)

payload = {"rows": rows, "stats": stats}
with open(summary_json_path, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)

payload["stats"]


In [ ]:
summary_csv_path, summary_json_path, summary_csv_path.exists(), summary_json_path.exists()


In [ ]:
summary_rows_preview = []
if summary_csv_path.exists():
    with open(summary_csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        for i, row in enumerate(reader):
            summary_rows_preview.append(row)
            if i >= 10:
                break

summary_rows_preview


In [ ]:
{
    "summary_csv": str(summary_csv_path),
    "summary_json": str(summary_json_path),
    "rows_count": len(rows),
    "stats": stats,
    "enriched_files_count": len(enriched_files),
    "enriched_files_preview": [str(path) for path in enriched_files[:3]],
}
